In [51]:
from pyspark.sql import SparkSession, Row
import requests
import json
import datetime
import os

# Inicializa a sessão PySpark
spark = SparkSession.builder.appName("SteamGames").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")  # Ajuste o nível de log para ERROR


In [52]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

with open(dir_path + '/steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [53]:
# Verificar o ambiente
if "dev" in dir_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [54]:
# Função para obter os ids dos jogos
def get_game_ids(api_key, steam_id):
    url = f'http://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/?key={api_key}&steamid={steam_id}&format=json'
    response = requests.get(url)
    data = response.json()
    games = data['response']['games']

    return games

In [55]:
# Função para obter os detalhes dos jogos
def get_game_details(games):

    game_user_list = []    
    
    for game in games:
        appid = game['appid']
        app_details_response = requests.get(f'https://store.steampowered.com/api/appdetails?appids={appid}')
        app_details_data = app_details_response.json()
        game_user_list.append(app_details_data)

    return game_user_list

In [56]:
# Função para processar os detalhes dos jogos e criar um dataframe
def process_game_from_user(game_user_list):
    game_data_user = []

    for response in game_user_list:
        for app_id, data in response.items():
            if 'data' in data:
                game_data = {
                    'steam_appid': data['data'].get('steam_appid'),
                    'type': data['data'].get('type'),
                    'name': data['data'].get('name'),
                    'required_age': data['data'].get('required_age'),
                    'is_free': data['data'].get('is_free'),
                    'header_image': data['data'].get('header_image'),
                    'website': data['data'].get('website'),
                    'pc_requirements': data['data'].get('pc_requirements'),
                    'package_groups': data['data'].get('package_groups'),
                    'short_description': data['data'].get('short_description')
                }
                game_data_user.append(game_data)

    return game_data_user

In [57]:
# Substitua suas credenciais aqui
API_KEY = config["API_KEY"]
STEAM_ID = config["STEAM_ID"]

#Obtenha os detalhes dos jogos
games = get_game_ids(API_KEY, STEAM_ID)

In [58]:
print(games)

[{'appid': 4700, 'playtime_forever': 999, 'playtime_windows_forever': 999, 'playtime_mac_forever': 0, 'playtime_linux_forever': 0, 'playtime_deck_forever': 0, 'rtime_last_played': 1693270855, 'playtime_disconnected': 0}, {'appid': 4760, 'playtime_forever': 46007, 'playtime_windows_forever': 46007, 'playtime_mac_forever': 0, 'playtime_linux_forever': 0, 'playtime_deck_forever': 0, 'rtime_last_played': 1670092988, 'playtime_disconnected': 0}, {'appid': 4770, 'playtime_forever': 6, 'playtime_windows_forever': 6, 'playtime_mac_forever': 0, 'playtime_linux_forever': 0, 'playtime_deck_forever': 0, 'rtime_last_played': 1588639205, 'playtime_disconnected': 0}, {'appid': 214950, 'playtime_forever': 29589, 'playtime_windows_forever': 29589, 'playtime_mac_forever': 0, 'playtime_linux_forever': 0, 'playtime_deck_forever': 0, 'rtime_last_played': 1727635769, 'playtime_disconnected': 0}, {'appid': 271590, 'playtime_forever': 3668, 'playtime_windows_forever': 3668, 'playtime_mac_forever': 0, 'playtim

In [59]:
# Obtenha os detalhes dos jogos
game_user_list = get_game_details(games)

In [60]:
print(game_user_list)

[{'4700': {'success': True, 'data': {'type': 'game', 'name': 'Total War: MEDIEVAL II – Definitive Edition', 'steam_appid': 4700, 'required_age': 0, 'is_free': False, 'detailed_description': '<img class="bb_img" src="https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/4700/extras/MED2_FIRST.gif?t=1724863909" /><br><br>Complete your Total War collection with this Definitive Edition of Total War: MEDIEVAL II, which includes all DLC and feature updates since the game’s release: <br><br>Kingdoms is the most content-rich expansion ever produced for a Total War game, with four new entire campaigns centred on expanded maps of the British Isles, Teutonic Northern Europe, the Middle East, and the Americas.<br><br>Total War: MEDIEVAL II Definitive Edition offers hundreds and hundreds of hours of absorbing gameplay and every bit of content made for the game. See below for full details. <br><br><img class="bb_img" src="https://shared.akamai.steamstatic.com/store_item_assets/steam/app

In [61]:
# Processe os dados dos jogos do usuario
game_data_user = process_game_from_user(game_user_list)

In [62]:
print(game_data_user)

[{'steam_appid': 4700, 'type': 'game', 'name': 'Total War: MEDIEVAL II – Definitive Edition', 'required_age': 0, 'is_free': False, 'header_image': 'https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/4700/header.jpg?t=1724863909', 'website': 'http://www.totalwar.com', 'pc_requirements': {'minimum': '<strong>Minimum System Requirements:</strong><br>\t\t\t\t\t<ul class="bb_ul"><li>Microsoft® Windows® 2000/XP<br>\t\t\t\t\t</li><li>Celeron 1.5GHz Pentium 4® (1500MHz) or equivalent AMD® processor<br>\t\t\t\t\t</li><li>512MB RAM<br>\t\t\t\t\t</li><li>11GB of uncompressed free hard disk space<br>\t\t\t\t\t</li><li>100% DirectX® 9.0c compatible 16-bit sound cardand latest drivers<br>\t\t\t\t\t</li><li>100% Windows® 2000/XP compatible mouse,keyboard and latest drivers<br>\t\t\t\t\t</li><li>DirectX® 9.0c<br>\t\t\t\t\t</li><li>128MB Hardware Accelerated video card with Shader 1 support and the latest drivers. Must be 100% DirectX® 9.0c compatible<br>\t\t\t\t\t</li><li>1024 x 768 mi

In [63]:
# Crie um dataframe PySpark a partir dos dados processados
games_df = spark.createDataFrame(games)
final_data_user_df = spark.createDataFrame([Row(**data) for data in game_data_user])
df = games_df.join(final_data_user_df, games_df.appid == final_data_user_df.steam_appid, 'inner').drop("steam_appid")
    

# Mostre o dataframe
df.show()

+-------+---------------------+---------------------+----------------+----------------------+--------------------+------------------------+-----------------+---------------+----+--------------------+------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|  appid|playtime_deck_forever|playtime_disconnected|playtime_forever|playtime_linux_forever|playtime_mac_forever|playtime_windows_forever|rtime_last_played|playtime_2weeks|type|                name|required_age|is_free|        header_image|             website|     pc_requirements|      package_groups|   short_description|
+-------+---------------------+---------------------+----------------+----------------------+--------------------+------------------------+-----------------+---------------+----+--------------------+------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|    730|             

In [64]:
# Salve o dataframe como um único arquivo JSON
current_date_str = datetime.datetime.now().strftime("%Y-%m-%d")
base_path = dir_path + "/datalake/steam/user/inbound"
os.makedirs(base_path, exist_ok=True)
temp_path = os.path.join(base_path, "temp_json")

# Coalesce para garantir que seja salvo em um único arquivo
df.coalesce(1).write.json(temp_path, mode="overwrite")

# Renomeie o arquivo salvo
for file_name in os.listdir(temp_path):
    if file_name.endswith(".json"):
        os.rename(os.path.join(temp_path, file_name), os.path.join(base_path, f"{current_date_str}.json"))

# Remova todos os arquivos na pasta temporária
for file_name in os.listdir(temp_path):
    os.remove(os.path.join(temp_path, file_name))

# Remova o diretório temporário
os.rmdir(temp_path)